# Битва за партицию

Представим, что у нас есть таблица inventory с остатками товаров.

Джоб А (Склад): Обновляет остатки (stock).

Джоб Б (Маркетинг): Применяет скидки (discount).

Оба джоба пишут в одну партицию dt='2025-09-04' используя INSERT OVERWRITE.

In [ ]:
%run /home/jovyan/work/init_spark.py yarn

In [4]:
# 1. Инициализация "болота"
spark.sql("DROP TABLE IF EXISTS race_test_table")
spark.sql("""
    CREATE TABLE race_test_table (
        item_id INT, 
        stock INT, 
        discount DECIMAL(5,2)
    ) 
    PARTITIONED BY (dt STRING)
    STORED AS PARQUET
""")
# Начальные данные
spark.sql("INSERT INTO race_test_table PARTITION (dt='2025-09-04') VALUES (101, 100, 0.0)")

print("=== СТАРТ: Изначальное состояние ===")
spark.sql("SELECT * FROM race_test_table WHERE dt='2025-09-04'").show()

=== СТАРТ: Изначальное состояние ===
+-------+-----+--------+----------+
|item_id|stock|discount|        dt|
+-------+-----+--------+----------+
|    101|  100|    0.00|2025-09-04|
+-------+-----+--------+----------+



## Сценарий 1: Оба джоба успешны (но данные дублируются/перетираются)

In [5]:
print("\n--- Сценарий 1: Параллельная запись (Race Condition) ---")

# Эмуляция Джоба А (Склад)
df_job_a = spark.createDataFrame([(101, 50)], ["item_id", "stock"])
df_job_a.write.mode("overwrite").insertInto("race_test_table")

# Эмуляция Джоба Б (Маркетинг) - запускается почти одновременно
df_job_b = spark.createDataFrame([(101, 0, 0.50)], ["item_id", "stock", "discount"]) # Упрощено для примера
# В реальности это разные схемы, но Hive добавит NULL в отсутствующие колонки
df_job_b.write.mode("overwrite").insertInto("race_test_table")

print("Результат в Hive (после обоих джобов):")
spark.sql("SELECT * FROM race_test_table WHERE dt='2025-09-04'").show()


--- Сценарий 1: Параллельная запись (Race Condition) ---


AnalysisException: [INSERT_COLUMN_ARITY_MISMATCH.NOT_ENOUGH_DATA_COLUMNS] Cannot write to `iceberg`.`race_test_table`, the reason is not enough data columns:
Table columns: `item_id`, `stock`, `discount`, `dt`.
Data columns: `item_id`, `stock`.

> Внимание! В Hive могут появиться строки с NULL значениями или дубликаты, если файловый менеджер успел их прочитать.